In [0]:
from pyspark.sql import functions
from pyspark.sql.types import StructType, StructField, StringType
spark.sql("USE CATALOG Capstone_team01")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("USE SCHEMA bronze")

In [0]:
volume_root = "/Volumes/capstone_team01/bronze/raw_data/"

transaction_path = volume_root + "datasets/transaction_data/card_transactions/"

authentication_path = volume_root + "datasets/event_data/device_authentication_events.csv"

payment_changes_path = volume_root + "datasets/cdc_data/payment_status_changes.csv"

metadata_path = volume_root + "autoloader_metadata/"
transaction_schema_location = metadata_path + "card_transactions/schema/"
transaction_checkpoint = metadata_path + "card_transactions/checkpoint/"

In [0]:
from pyspark.sql.types import StructType, StructField
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("card_id", StringType(), True),
    StructField("transaction_ts", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("country", StringType(), True),
    StructField("authorization_code", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("device_id", StringType(), True)
])

In [0]:
transaction_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", transaction_schema_location)
    .option("rescuedDataColumn", "_rescued_data")
    .schema(transaction_schema)
    .load(transaction_path)
)

transaction_query = (
    transaction_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", transaction_checkpoint)
    .trigger(availableNow=True)
    .toTable("Capstone_team01.bronze.card_transactions")
)

transaction_query.awaitTermination()

In [0]:
authentication_schema = StructType([
    StructField("auth_event_id", StringType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("authentication_method", StringType(), True),
    StructField("authentication_result", StringType(), True),
    StructField("ip_country", StringType(), True),
    StructField("attempt_number", StringType(), True),
    StructField("risk_signal", StringType(), True)
])

In [0]:
authentication_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(authentication_schema)
    .load(authentication_path)
)

(
    authentication_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Capstone_team01.bronze.device_authentication_events")
)

In [0]:
payment_changes_schema = StructType([
    StructField("change_event_id", StringType(), True),
    StructField("transaction_id", StringType(), True),
    StructField("change_ts", StringType(), True),
    StructField("old_status", StringType(), True),
    StructField("new_status", StringType(), True),
    StructField("reason_code", StringType(), True),
    StructField("change_sequence", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("operator_id", StringType(), True),
    StructField("correlation_id", StringType(), True)
])

In [0]:
payment_changes_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(payment_changes_schema)
    .load(payment_changes_path)
)

(
    payment_changes_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Capstone_team01.bronze.payment_status_changes")
)

In [0]:
sql_server = "capstone1dbserver.database.windows.net"
sql_database = "capstone1team1creditcard"

sql_username = dbutils.secrets.get(
    scope="azure-sql-scope",
    key="sql-user"
)

sql_password = dbutils.secrets.get(
    scope="azure-sql-scope",
    key="sql-pwd"
)

jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

connection_properties = {
    "user": sql_username,
    "password": sql_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
customer_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.customer_master",
    properties=connection_properties
)

customer_bronze_df = customer_df.select(
    *[
        functions.col(c).cast("string").alias(c)
        for c in customer_df.columns
    ]
)

(
    customer_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Capstone_team01.bronze.customer_master")
)

In [0]:
merchant_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.merchant_master",
    properties=connection_properties
)

merchant_bronze_df = merchant_df.select(
    *[
        functions.col(c).cast("string").alias(c)
        for c in merchant_df.columns
    ]
)

(
    merchant_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Capstone_team01.bronze.merchant_master")
)

In [0]:
card_df = spark.read.jdbc(
    url=jdbc_url,
    table="dbo.card_master",
    properties=connection_properties
)

card_bronze_df = card_df.select(
    *[
        functions.col(c).cast("string").alias(c)
        for c in card_df.columns
    ]
)

(
    card_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Capstone_team01.bronze.card_master")
)

In [0]:
bronze_tables = [
    "card_transactions",
    "device_authentication_events",
    "payment_status_changes",
    "customer_master",
    "merchant_master",
    "card_master"
]

for table_name in bronze_tables:

    full_table_name = f"Capstone_team01.bronze.{table_name}"

    df = spark.table(full_table_name)

    print("=" * 60)
    print("TABLE:", full_table_name)
    print("TOTAL RECORDS:", df.count())

    df.printSchema()

    display(df.limit(5))